# Full Parameter Fine-Tuning

A practical refresher on **full parameter fine-tuning (full FT)** — continuing the
training of a pre-trained model while updating **every weight** in the network, as
opposed to parameter-efficient methods (LoRA/QLoRA, adapters, prompt tuning) that freeze
the base and train a small add-on.

Full FT is the most expressive and most expensive way to adapt a model. It can move
behavior the furthest from the base, but it costs the most memory, the most compute, and
produces a full-size checkpoint per task. This notebook is written from a **DevOps/MLOps
infrastructure** angle: the memory math, the distributed-training stack (FSDP, DeepSpeed
ZeRO), checkpointing, throughput, cost, and how to operate it in production.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction <a id="introduction"></a>

Full parameter fine-tuning takes a pre-trained checkpoint and continues gradient descent
with **all parameters trainable**. Every weight matrix — attention projections, MLP
layers, embeddings, layer norms — receives gradients and is updated by the optimizer.
The training objective is usually the same next-token cross-entropy used in pre-training,
but on a curated, often labeled dataset, with the loss commonly **masked to the
completion** for instruction/chat data.

### What is it?

Mechanically it is ordinary supervised training. You load weights `theta_0`, run forward
and backward passes over your dataset, and update `theta <- theta - lr * grad L(theta)`
for the **entire** weight vector `theta`. The output is a brand-new full checkpoint the
same size as the base. With LoRA you would instead freeze `theta_0` and learn a small
`dW = B*A`; full FT has no such constraint, so it can represent any change the
architecture allows.

### Why use it?

Key benefits of using full parameter fine-tuning:

- **Maximum capacity to change behavior.** No low-rank bottleneck — useful for large
  domain shifts, new languages, new output formats, or distillation where adapters
  underfit.
- **Often the best quality at convergence** when you have enough high-quality data and
  compute; LoRA can match it on narrow tasks but tends to trail on broad, multi-task or
  large-shift objectives.
- **Single self-contained artifact.** The result is a normal model you can quantize,
  convert (GGUF/AWQ), and serve with any runtime — no adapter-merging step at load time.

### When to use it?

Full FT is particularly worth its cost when:

- You have a **large, high-quality dataset** (tens of thousands to millions of examples)
  and the target differs substantially from the base model's behavior.
- You need to **bake in a new modality, language, or format** that low-rank adapters
  struggle to capture.
- You will **serve one specialized model heavily**, so amortizing one expensive training
  run over huge inference volume is fine, rather than juggling many cheap per-task
  adapters.

## Key Features <a id="key-features"></a>

### What distinguishes full parameter fine-tuning

| Capability | What it means | Benefit / cost |
|------------|---------------|----------------|
| **All weights trainable** | Optimizer state exists for every parameter; no frozen base | Maximum expressiveness; highest memory and compute |
| **Optimizer-state dominated memory** | Adam keeps fp32 master weights + 2 moment buffers, ~4-6x the raw weight bytes | Drives the need for sharding (ZeRO/FSDP) past ~3B params |
| **Sharded data parallelism** | FSDP / DeepSpeed ZeRO-2/3 split params, grads, and optimizer state across GPUs | Lets a 7B-70B+ model fit by trading communication for memory |
| **Activation checkpointing** | Recompute activations in backward instead of storing them | Cuts activation memory ~sqrt(depth) at ~20-30% compute overhead |
| **Full-size checkpoint output** | Artifact equals the base model size (~14 GB for a 7B in bf16) | Portable, quantizable, servable anywhere; storage-heavy per task |
| **Sensitive to LR and data** | Touching every weight makes forgetting and overfitting easy | Needs small LR, warmup, and held-out eval to stay safe |

## Architecture Overview <a id="architecture"></a>

The model and loss are unchanged from pre-training — the same transformer, the same
causal-LM objective. What changes is that **nothing is frozen** and the **memory budget
explodes**, which forces a distributed-training architecture.

```
  Base checkpoint theta_0 --> [ Sharded across N GPUs via FSDP / ZeRO-3 ]
        |
        v
  +----------------------- per training step ------------------------+
  |  all-gather shard -> forward (activation checkpointing) -> loss  |
  |  -> backward -> reduce-scatter grads -> optimizer.step (sharded) |
  +-----------------------------------------------------------------+
        |
        v
  Full checkpoint theta*  (same size as theta_0; consolidated from shards)
```

### Components

1. **Trainer / training loop**: Hugging Face `Trainer`/`SFTTrainer`, PyTorch Lightning,
   or a custom loop driving forward/backward/step over the dataset.
2. **Sharding engine**: **FSDP** (native PyTorch) or **DeepSpeed ZeRO** partitions
   parameters, gradients, and optimizer state to fit large models across GPUs; ZeRO-Offload
   can spill optimizer state to CPU/NVMe.
3. **Optimizer + precision**: AdamW with mixed precision (bf16 compute, fp32 master
   weights). Optimizer state — not the weights — is usually the largest memory consumer.
4. **Memory savers**: activation/gradient checkpointing, fused/8-bit optimizers
   (`bitsandbytes` `adamw_8bit`), and gradient accumulation to reach a large effective
   batch on limited VRAM.
5. **Checkpoint manager**: writes sharded checkpoints periodically and consolidates them
   into a standard `from_pretrained`-loadable model at the end.

## Installation <a id="installation"></a>

### Prerequisites

- **GPU memory is the gating resource.** A useful rule of thumb for AdamW mixed-precision
  full FT is **~16-20 GB of GPU memory per billion parameters** (fp32 master weights +
  bf16 weights + fp32 Adam m/v + gradients + activations). So a 7B model wants
  ~**112-140 GB** — multiple GPUs with FSDP/ZeRO, not a single 24 GB card.
- **CUDA 12.x**, a recent NVIDIA driver, and PyTorch 2.x with a matching CUDA build.
- For multi-GPU/multi-node: a fast interconnect (NVLink within a node, InfiniBand/EFA
  across nodes) — full FT is communication-heavy because grads/params are sharded.
- Core libraries: `transformers`, `trl`, `datasets`, `accelerate`, and either
  `deepspeed` or PyTorch-native FSDP. `bitsandbytes` enables 8-bit Adam to cut optimizer
  memory; `flash-attn` speeds up attention.

### Installation Steps

**Note**: Uncomment the following cell to install in Google Colab or a fresh environment.

In [ ]:
# Uncomment to install the full-parameter fine-tuning stack
# !pip install -U torch transformers trl datasets accelerate
# !pip install -U deepspeed bitsandbytes      # ZeRO sharding + 8-bit Adam
# !pip install -U flash-attn --no-build-isolation   # optional: faster attention

# Quick sanity check of the environment
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print("GPUs:", n)
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

## Basic Usage <a id="basic-usage"></a>

### Quick Start Example

The minimal full-FT loop with Hugging Face `SFTTrainer`. The only thing that makes this
*full* fine-tuning is that we **do not** attach a `peft_config` / LoRA — every parameter
stays trainable. We train in bf16 with gradient checkpointing to keep memory in check.

In [ ]:
# Full-parameter SFT of a small base model on a chat dataset.
# (Run on a GPU; the model below is intentionally tiny so it fits a single card.)
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

model_name = "Qwen/Qwen2.5-0.5B"          # small base so full FT fits one GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="bfloat16",
    attn_implementation="flash_attention_2",  # drop if flash-attn isn't installed
)

# Confirm this is FULL fine-tuning: every parameter requires grad.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")

dataset = load_dataset("trl-lib/Capybara", split="train[:2000]")

config = SFTConfig(
    output_dir="qwen-0.5b-fullft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,            # effective batch = 16 per device
    learning_rate=1e-5,                       # small LR: we touch every weight
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,              # trade compute for activation memory
    logging_steps=10,
    save_strategy="epoch",
)

trainer = SFTTrainer(model=model, args=config, train_dataset=dataset)
trainer.train()
trainer.save_model("qwen-0.5b-fullft")        # writes a standard, full-size checkpoint
tokenizer.save_pretrained("qwen-0.5b-fullft")

## Advanced Features <a id="advanced-features"></a>

### Scaling full FT past a single GPU

#### Feature 1: Estimating whether the model even fits

Before launching, do the memory math. For AdamW mixed precision the per-parameter cost is
roughly 2 bytes bf16 weight + 4 bytes fp32 master + 4 bytes grad + 4+4 bytes Adam m/v
~= **18 bytes/param**, plus activations. The helper below turns that into a per-GPU
estimate so you can pick a sharding strategy (single GPU, FSDP/ZeRO-3, or ZeRO-3 with
offload).

In [ ]:
def fullft_memory_gb(num_params_b, n_gpus=1, bytes_per_param=18, zero_stage=3,
                     activation_gb=10):
    """Rough GPU-memory estimate for AdamW mixed-precision full fine-tuning.

    num_params_b   : model size in billions of params
    n_gpus         : data-parallel world size
    bytes_per_param: ~18 = 2(bf16) + 4(fp32 master) + 4(grad) + 8(Adam m+v)
    zero_stage     : 2 = shard grads+optim, 3 = shard params+grads+optim
    activation_gb  : per-GPU activation budget (depends on seq len, batch, ckpting)
    """
    model_state = num_params_b * 1e9 * bytes_per_param
    if zero_stage == 3:
        per_gpu = model_state / n_gpus            # everything sharded evenly
    elif zero_stage == 2:
        weights = num_params_b * 1e9 * 2          # bf16 weights replicated
        per_gpu = weights + (model_state - weights) / n_gpus
    else:                                          # stage 0/1: no param/grad shard
        per_gpu = model_state
    return per_gpu / 1e9 + activation_gb


for params_b in (0.5, 7, 13, 70):
    for gpus in (1, 8):
        gb = fullft_memory_gb(params_b, n_gpus=gpus, zero_stage=3)
        verdict = "fits 80GB" if gb <= 80 else "needs offload / more GPUs"
        print(f"{params_b:>4}B  x{gpus} GPU (ZeRO-3): ~{gb:6.1f} GB/GPU  -> {verdict}")

## Use Cases <a id="use-cases"></a>

### Real-world applications of full parameter fine-tuning

#### Use Case 1: Instruction-tuning a base model into a chat assistant

- **Context**: You have a raw base model (next-token only) and a large curated SFT mixture
  (hundreds of thousands of chat turns) and want a general-purpose assistant.
- **Implementation**: Full FT across a multi-GPU node with FSDP/ZeRO-3, bf16, packed
  sequences, cosine LR ~1e-5 to 2e-5, 1-3 epochs, held-out eval each epoch.
- **Results**: Broad behavior change LoRA tends to underfit; the output is a standard
  checkpoint ready for quantization and serving.

#### Use Case 2: New-language / large-domain-shift adaptation

- **Context**: Adapting an English-centric model to a low-resource language, or to a
  domain whose tokens/format are far from the base distribution (e.g. genomic sequences,
  a proprietary DSL).
- **Implementation**: Optionally extend the tokenizer/embeddings, then full FT (often
  after continued pre-training) so embedding and deep layers can move together.
- **Results**: Captures shifts a low-rank `dW` cannot, at the cost of a full training run
  and a full checkpoint per target.

#### Use Case 3: Distillation into a smaller serving model

- **Context**: Compress a large teacher's behavior into a smaller student you can serve
  cheaply at high QPS.
- **Implementation**: Full FT the student on teacher-generated data (and/or logits),
  where every student weight must adapt to absorb the teacher signal.
- **Results**: A small, fully-owned model whose one-time training cost is amortized over
  large inference volume.

## Best Practices <a id="best-practices"></a>

### Recommended practices for full parameter fine-tuning

1. **Use a small learning rate with warmup.** Typical full-FT LRs are **1e-5 to 2e-5**
   (often 5-10x smaller than LoRA), with 3-5% warmup and cosine decay — large LRs on every
   weight wreck the base model fast.
2. **Always keep a held-out eval set and a general-capability probe.** Full FT forgets
   easily; track both task metrics and a small "did it stay smart" benchmark each epoch.
3. **Shard before you offload.** Reach for FSDP/ZeRO-3 first; only add CPU/NVMe offload
   when sharding alone won't fit, since offload is much slower.
4. **Maximize effective batch with accumulation + packing.** Gradient accumulation and
   sequence packing stabilize training and improve GPU utilization without more VRAM.
5. **Checkpoint often and consolidate at the end.** Save sharded checkpoints on a cadence
   (preemption-safe), then write one standard `from_pretrained`-loadable model.
6. **Prefer bf16 over fp16.** bf16's wider exponent avoids the loss-scaling instabilities
   fp16 suffers in long full-FT runs; keep fp32 master weights either way.
7. **Mix in general data to fight forgetting.** Blending a few percent of general/replay
   data into the fine-tune set preserves base capabilities.

## Common Pitfalls <a id="pitfalls"></a>

### What to avoid when using full parameter fine-tuning

1. **Underestimating memory (OOM at step 0 or step N).** People budget for weights but
   forget optimizer state is ~2x the weights and activations grow with batch x seqlen. Use
   the memory estimator, enable activation checkpointing, and shard optimizer state.
2. **Catastrophic forgetting.** A too-high LR or too many epochs on a narrow dataset makes
   the model great at your task and worse at everything else. Use small LR, fewer epochs,
   replay data, and a general benchmark as a guardrail.
3. **fp16 loss-scaling blowups.** Long full-FT runs in fp16 can diverge/NaN from gradient
   underflow or scaler thrash. Prefer bf16; if stuck on fp16, watch the loss scale.
4. **Treating full FT as the default.** It's the most expensive option. If LoRA/QLoRA
   reaches your quality bar (it often does for narrow tasks), the per-task storage and
   compute savings are large — only pay for full FT when you've shown you need it.

## Performance Optimization <a id="performance"></a>

### Optimizing full parameter fine-tuning for throughput and memory

#### Configuration tuning

Key knobs and what they trade:

- **Sharding strategy (ZeRO-2 vs ZeRO-3 / FSDP `FULL_SHARD`)**: ZeRO-3 shards params too
  (lowest memory, most comm); ZeRO-2 shards only grads+optim (more memory, less comm). Pick
  the lightest that fits.
- **Activation/gradient checkpointing**: Big activation-memory win for ~20-30% extra
  compute; essential at long sequence lengths.
- **8-bit Adam (`optim="adamw_bnb_8bit"`)**: Cuts optimizer state from 8 to 2 bytes/param,
  often the cheapest way to fit without adding GPUs.
- **Gradient accumulation + sequence packing**: Raise effective batch and GPU utilization
  without extra VRAM; packing removes padding waste.
- **`torch.compile` and flash attention**: Kernel fusion and memory-efficient attention
  boost tokens/sec on Ampere/Hopper GPUs.

The cell below writes a DeepSpeed ZeRO-3 config and prints the `accelerate launch` command
— the common way to actually run multi-GPU full FT.

In [ ]:
import json

# DeepSpeed ZeRO-3 config tuned for full-parameter fine-tuning.
zero3 = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        # Spill optimizer state to CPU only if GPU memory is still short:
        "offload_optimizer": {"device": "none"},   # set to "cpu" to enable offload
        "stage3_gather_16bit_weights_on_model_save": True,  # consolidate on save
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": 1.0,
    "train_micro_batch_size_per_gpu": "auto",
}

with open("ds_zero3.json", "w") as f:
    json.dump(zero3, f, indent=2)
print("wrote ds_zero3.json")

# Launch full FT across all visible GPUs (run from a shell, not in-notebook):
launch_cmd = r"""
accelerate launch \
  --num_processes 8 --multi_gpu \
  --mixed_precision bf16 \
  train_sft.py \
  --model_name meta-llama/Llama-3.1-8B \
  --deepspeed ds_zero3.json \
  --gradient_checkpointing --optim adamw_bnb_8bit \
  --learning_rate 1e-5 --warmup_ratio 0.03 --num_train_epochs 3
""".strip()
print(launch_cmd)

## Production Deployment <a id="deployment"></a>

### Deploying a fully fine-tuned model

The output of full FT is a **standard checkpoint** — no adapter to merge — so it deploys
like any base model. The usual path is to (optionally) quantize, then serve with a
high-throughput engine such as vLLM or TGI behind an autoscaled service.

#### Docker Deployment

```dockerfile
# Serve a fully fine-tuned model with vLLM (OpenAI-compatible API)
FROM vllm/vllm-openai:latest
# Bake the consolidated checkpoint into the image, or mount it at runtime
COPY ./llama3-8b-fullft /models/llama3-8b-fullft
EXPOSE 8000
ENTRYPOINT ["python", "-m", "vllm.entrypoints.openai.api_server", \
            "--model", "/models/llama3-8b-fullft", \
            "--dtype", "bfloat16", "--max-model-len", "8192"]
```

#### Kubernetes Deployment

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: llama3-8b-fullft
spec:
  replicas: 2
  selector:
    matchLabels: {app: llama3-8b-fullft}
  template:
    metadata:
      labels: {app: llama3-8b-fullft}
    spec:
      containers:
        - name: vllm
          image: registry.example.com/llama3-8b-fullft:latest
          args: ["--model", "/models/llama3-8b-fullft", "--dtype", "bfloat16"]
          ports: [{containerPort: 8000}]
          resources:
            limits:
              nvidia.com/gpu: "1"           # one A100/H100 per replica
          readinessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 60
---
apiVersion: v1
kind: Service
metadata:
  name: llama3-8b-fullft
spec:
  selector: {app: llama3-8b-fullft}
  ports: [{port: 80, targetPort: 8000}]
```

Unlike LoRA serving, you can't hot-swap many task adapters onto one base — each fully
fine-tuned model is its own full-size deployment, so plan GPU capacity per model.

## Monitoring and Observability <a id="monitoring"></a>

### Monitoring full parameter fine-tuning and the served model

#### Key Metrics to Track — during training

- **Training & validation loss / perplexity**: The primary convergence signal; watch for
  val loss flattening or rising (overfitting) while train loss keeps dropping.
- **Gradient norm**: Spikes precede divergence; with bf16 a stable grad-norm under your
  clip threshold (e.g. 1.0) is a good health signal.
- **Throughput (tokens/sec, MFU)**: Tokens-per-second per GPU and model-FLOPs-utilization
  tell you if sharding/comm is bottlenecking you.
- **GPU memory, utilization, and power**: Headroom check and early warning for OOM as
  sequence lengths or batch sizes change.
- **General-capability eval**: A small periodic benchmark to catch catastrophic forgetting
  that loss alone won't show.

#### Key Metrics to Track — when served

- Request latency (TTFT, inter-token), throughput (req/s, tokens/s), GPU utilization, and
  output-quality / safety metrics.

#### Logging Best Practices

- Stream training metrics to **TensorBoard / Weights & Biases / MLflow**, log the full
  config + dataset hash + git SHA for reproducibility, and emit structured logs at sane
  levels so checkpoints and runs are traceable end to end.

## Troubleshooting <a id="troubleshooting"></a>

### Common issues with full parameter fine-tuning

#### Issue 1: CUDA out-of-memory

**Symptoms**: `CUDA out of memory` at the first step, or partway through when sequence
lengths grow.

**Cause**: Optimizer state (~8 bytes/param for Adam) plus activations exceed VRAM —
full FT's footprint is dominated by optimizer state, not weights.

**Solution**: Enable gradient checkpointing, switch to ZeRO-3 / FSDP `FULL_SHARD`, use
8-bit Adam, lower micro-batch and raise gradient accumulation, shorten `max_seq_len`, and
only then add CPU/NVMe optimizer offload.

#### Issue 2: Loss diverges / goes to NaN

**Symptoms**: Loss spikes, becomes `nan`, or the model collapses after a few hundred
steps.

**Cause**: Learning rate too high for full FT, fp16 loss-scaling underflow, or a missing
warmup so the first updates are too aggressive on every weight.

**Solution**: Drop LR to ~1e-5, add 3-5% warmup with cosine decay, switch fp16 to bf16,
and enable gradient clipping (max norm 1.0). Inspect grad-norm logs to find the step it
breaks.

#### Issue 3: Model forgets general abilities

**Symptoms**: Great on the fine-tune task, noticeably worse on general reasoning/chat.

**Cause**: Too-high LR or too many epochs on a narrow dataset overwrote pre-trained
knowledge (catastrophic forgetting).

**Solution**: Fewer epochs, smaller LR, blend in replay/general data, and gate releases on
a general-capability benchmark, not just task loss.

## Comparison with Alternatives <a id="comparison"></a>

### How full parameter fine-tuning compares to other adaptation methods

| Dimension | Full FT | LoRA / QLoRA | Prompt / prefix tuning | RAG (no training) |
|-----------|---------|--------------|------------------------|-------------------|
| Trainable params | 100% | ~0.1-1% | <0.1% | 0% |
| GPU memory to train | Highest (~16-20 GB/B) | Low-moderate | Lowest | None |
| Capacity to change behavior | Highest | High (narrow tasks) | Low | Adds knowledge, not behavior |
| Artifact size per task | Full model | Tens of MB adapter | Tiny | None |
| Multi-task serving | One model each | Many adapters on one base | Many prompts | One pipeline |
| Catastrophic-forgetting risk | Highest | Lower (base frozen) | Lowest | None |
| Best when | Large data + large shift | Most narrow adaptations | Lightweight steering | Fresh/changing facts |

### When to Choose This Approach

Choose full parameter fine-tuning when:

- You have a **large, high-quality dataset** and need a **large behavior/format/language
  shift** that low-rank adapters underfit.
- You'll **serve one specialized model at scale**, so a single expensive training run is
  easily amortized.
- You want a **self-contained checkpoint** to quantize/convert/distill, with no
  adapter-merge step — and you've confirmed PEFT doesn't already meet the bar.

## Resources <a id="resources"></a>

### Official Documentation

- Hugging Face TRL (`SFTTrainer`): https://huggingface.co/docs/trl
- Transformers `Trainer`: https://huggingface.co/docs/transformers/main_classes/trainer
- DeepSpeed (ZeRO): https://www.deepspeed.ai/
- PyTorch FSDP: https://pytorch.org/docs/stable/fsdp.html

### Tutorials and Guides

- Hugging Face — Fine-tuning & SFT: https://huggingface.co/docs/transformers/training
- DeepSpeed ZeRO tutorial: https://www.deepspeed.ai/tutorials/zero/
- PyTorch FSDP getting-started: https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html

### Key Papers

- ZeRO: Memory Optimizations Toward Training Trillion-Parameter Models (Rajbhandari et al., 2019): https://arxiv.org/abs/1910.02054
- Mixed Precision Training (Micikevicius et al., 2017): https://arxiv.org/abs/1710.03740
- LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021): https://arxiv.org/abs/2106.09685

### Related Technologies

- LoRA / QLoRA (PEFT) — the parameter-efficient alternative
- FSDP and DeepSpeed ZeRO — the sharding engines that make full FT fit
- vLLM / TGI — serving the resulting full checkpoint
- Flash Attention, `bitsandbytes` 8-bit Adam, activation checkpointing — memory/throughput tooling